# QLoRA SFT of Qwen2.5-VL-3B-Instruct on MechaVerify

This notebook runs the supervised fine-tuning experiment of the paper on a single 16 GB GPU. It regenerates the released v0.1 set from seeds, trains a QLoRA adapter on its train split, and scores the adapter with the deterministic verifier on the in-distribution, OOD and adversarial splits (56 items each, the same items for every model). The last section measures the perception gap (S minus D) of the base model and of the adapter.

Settings follow the paper. The base model is loaded in 4-bit NF4 with double quantization and fp16 compute. LoRA adapters of rank 16 (alpha 32, dropout 0.05) train on the language-model projections while the vision encoder stays frozen. The loss covers only the solver's `<think>/<answer>` rationales. Training runs 200 steps at learning rate 1e-4 with batch 1, gradient accumulation 4, paged AdamW-8bit and gradient checkpointing, and images are capped at 512·28·28 pixels. Evaluation is greedy with a 1536-token cap. The paper reports seeds 0, 1 and 2.

In [ ]:
SEED = 0        # 0, 1 or 2 (the paper reports all three)
SMOKE = False   # True runs a tiny end-to-end check instead of the full run

In [ ]:
# Use this repository when the notebook runs from a clone, otherwise clone it.
import os, sys, subprocess
def _repo_root():
    for d in (os.getcwd(), os.path.abspath(".."), os.path.abspath("MechaVerify")):
        if os.path.isfile(os.path.join(d, "mechaverify", "__init__.py")):
            return d
    return None
REPO = _repo_root()
if REPO is None:
    REPO = os.path.abspath("MechaVerify")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/itswaseemsajjad/MechaVerify.git", REPO], check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
%pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
import json, os
import torch
import transformers as _tf; _tf.set_seed(SEED)
from mechaverify.dataset import generate_dataset, default_plans, SplitPlan, read_manifest
from mechaverify.tasks import list_tasks
from mechaverify.train.data import to_sft_records, to_chat
from mechaverify.eval.runner import select_instances
from mechaverify.verify import verify

def log(*a): print("[mv]", *a, flush=True)

# The released v0.1 set (825 instances, seeds 1,000,000 to 1,000,780), regenerated from
# seeds. Training uses its train split. Every model is scored on the same fixed subset of
# its test splits: the first 8 items per task of id_test, ood_test and adversarial
# (n = 56 per split), the same items as `mechaverify evaluate --split <split> --per-task 8`.
if SMOKE:  # tiny end-to-end check
    plans = [SplitPlan("train", "mixed", [1, 2], 8),
             SplitPlan("id_test", "mixed", [1, 2], 3),
             SplitPlan("ood_test", "mixed", [5, 6], 3),
             SplitPlan("adversarial", "adv_idler", [2, 3], 3)]
    DATA, EVAL_PER_TASK = "data/mv_smoke", 2
else:
    plans = default_plans(60)
    DATA, EVAL_PER_TASK = "data/mv_release_v0_1", 8
card = generate_dataset(DATA, tasks=list_tasks(), plans=plans, seed0=1_000_000)
MANIFEST = DATA + "/manifest.jsonl"
if not SMOKE:  # the exact released set (manifest SHA-256 with LF line endings)
    import hashlib
    V01_SHA256 = "5db2d657facb6df25d3146f5e79298e58402821a46e291a4efe0c9fde762442b"
    _sha = hashlib.sha256(open(MANIFEST, "rb").read().replace(b"\r\n", b"\n")).hexdigest()
    assert card["n_instances"] == 825 and _sha == V01_SHA256, "the regenerated data differ from the released v0.1 set"
log("dataset:", card["n_instances"], card["counts_by_split"])
MAX_STEPS = 10 if SMOKE else 200
OUT = f"runs/{'smoke_' if SMOKE else ''}sft_s{SEED}"
os.makedirs(OUT, exist_ok=True)

In [ ]:
from transformers import (Qwen2_5_VLForConditionalGeneration, AutoProcessor,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# QLoRA: 4-bit NF4 with double quantization, fp16 compute (bf16 yields NaN generations
# on pre-Ampere GPUs). The same loading is used for training and for evaluation.
CDT = torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=CDT)
MAX_PIXELS = 512*28*28  # image resolution cap
# LoRA on the language-model projections only; the vision encoder stays frozen.
LORA_TARGETS = r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"
LORA = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM", target_modules=LORA_TARGETS)
LORA_PARAMS = 29_933_568  # rank-16 LoRA on the 36 language-model layers, nothing else
# evaluation: plain greedy decoding (no repetition penalty) with a 1536-token cap
MAX_NEW_TOKENS = 256 if SMOKE else 1536
GEN = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0)
log("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
    if torch.cuda.is_available() else "no GPU")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=CDT, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
_tf.set_seed(SEED)  # LoRA initialization
model = get_peft_model(model, LORA)
model.print_trainable_parameters()
n_lora = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert n_lora == LORA_PARAMS, f"{n_lora} trainable parameters, expected LoRA on the language model only"

In [ ]:
# Completion-only loss: everything up to the last assistant header is masked.
ASST = processor.tokenizer("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]
def _find(seq, sub):
    for i in range(len(seq) - len(sub), -1, -1):
        if seq[i:i+len(sub)] == sub:
            return i + len(sub)
    return 0
def collate(batch):
    msgs = [b["messages"] for b in batch]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in msgs]
    imgs, _ = process_vision_info(msgs)
    enc = processor(text=texts, images=imgs, padding=True, return_tensors="pt")
    labels = enc["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    it = getattr(model.config, "image_token_id", None)
    if it is not None:
        labels[labels == it] = -100
    for i in range(labels.size(0)):
        labels[i, :_find(enc["input_ids"][i].tolist(), ASST)] = -100
    enc["labels"] = labels
    return enc

records = to_sft_records(MANIFEST, split="train")
class DS(torch.utils.data.Dataset):
    def __len__(self): return len(records)
    def __getitem__(self, i): return records[i]
log("train examples:", len(records))

In [ ]:
args = TrainingArguments(
    output_dir=OUT, per_device_train_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=1e-4, max_steps=MAX_STEPS,
    fp16=True, bf16=False, gradient_checkpointing=True, optim="paged_adamw_8bit",
    logging_steps=5, save_strategy="no", report_to="none",
    remove_unused_columns=False, seed=SEED, data_seed=SEED)
Trainer(model=model, args=args, train_dataset=DS(), data_collator=collate).train()
model.save_pretrained(f"{OUT}/adapter")
processor.save_pretrained(f"{OUT}/adapter")
log("saved adapter ->", f"{OUT}/adapter")

In [ ]:
# Evaluate per split (greedy) with the deterministic verifier.
# Leave train mode and gradient checkpointing and re-enable the cache before
# generating, or the just-trained model yields NaN/garbage tokens.
model.eval()
try:
    model.gradient_checkpointing_disable()
except Exception as e:
    log("grad-ckpt disable:", e)
model.config.use_cache = True
@torch.no_grad()
def gen(inst):
    rec = to_chat(inst, DATA, include_target=False)
    text = processor.apply_chat_template(rec["messages"], tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(rec["messages"])
    enc = processor(text=[text], images=imgs, return_tensors="pt").to(model.device)
    out = model.generate(**enc, **GEN)
    return processor.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)[0]

results = {}
samples = []
allins = read_manifest(MANIFEST)
for split in ("id_test", "ood_test", "adversarial"):
    insts = select_instances(allins, split=split, per_task=EVAL_PER_TASK)
    correct = 0
    for i in insts:
        g = gen(i)
        vr = verify(i, g)
        correct += vr.correct
        if len(samples) < 10:  # keep a few raw generations for inspection
            samples.append({"task": i.task, "expected": str(i.answer.value),
                            "raw": (g or "")[:500], "parsed": str(vr.parsed_answer),
                            "format_ok": bool(vr.format_ok), "correct": bool(vr.correct)})
    acc = correct / len(insts) if insts else 0.0
    results[split] = {"acc": acc, "n": len(insts)}
    log(f"eval {split}: acc={acc:.3f} (n={len(insts)})")
for s in samples[:4]:
    log("SAMPLE", s["task"], "| exp:", s["expected"], "| raw:", repr(s["raw"][:160]))

with open(f"{OUT}/sft_eval.json", "w") as f:
    json.dump({"smoke": SMOKE, "seed": SEED, "model": MODEL_ID, "max_steps": MAX_STEPS,
               "dataset": card["counts_by_split"], "results": results,
               "samples": samples}, f, indent=2)
log("DONE ->", f"{OUT}/sft_eval.json")

## Perception gap of the base model and the SFT adapter

Every scored item is answered under condition D (the diagram) and condition S (the gold symbolic description, no image), with the same 4-bit loading and greedy decoding as above. The base-model D accuracies are the zero-shot row of the adaptation results. The base model is deterministic, so it is scored only in the `SEED = 0` run. The adapter is reattached to a freshly loaded base here, so its condition-D score can differ by an item or two from the evaluation above.

In [ ]:
import gc
from peft import PeftModel
from mechaverify.eval.prompts import SYSTEM, build_prompt

del model
gc.collect()
torch.cuda.empty_cache()
allins = read_manifest(MANIFEST)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=CDT, device_map="auto")
model.eval()
proc = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=MAX_PIXELS)

@torch.no_grad()
def gen_cond(inst, condition):
    if condition == "D":
        rec = to_chat(inst, DATA, include_target=False)
        msgs = rec["messages"]
        text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        imgs, _ = process_vision_info(msgs)
        enc = proc(text=[text], images=imgs, return_tensors="pt").to(model.device)
    else:  # S: symbolic description only, no image
        msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
                {"role": "user", "content": [{"type": "text", "text": build_prompt(inst, "S")}]}]
        text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = proc(text=[text], return_tensors="pt").to(model.device)
    out = model.generate(**enc, **GEN)
    return proc.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def eval_all(tag):
    res = {}
    for split in ("id_test", "ood_test", "adversarial"):
        insts = select_instances(allins, split=split, per_task=EVAL_PER_TASK)
        for cond in ("D", "S"):
            c = sum(verify(i, gen_cond(i, cond)).correct for i in insts)
            res[f"{split}_{cond}"] = round(c / len(insts), 4) if insts else 0.0
        res[f"{split}_gap"] = round(res[f"{split}_S"] - res[f"{split}_D"], 4)
        log(f"{tag} {split}: D={res[f'{split}_D']} S={res[f'{split}_S']} gap={res[f'{split}_gap']}")
    return res

perception = {"smoke": SMOKE, "seed": SEED}
if SEED == 0:  # the base model is deterministic, so one evaluation serves all seeds
    perception["base"] = eval_all("BASE")
model = PeftModel.from_pretrained(model, f"{OUT}/adapter")
model.eval()
perception["sft"] = eval_all("SFT")
with open(f"{OUT}/perception_eval.json", "w") as f:
    json.dump(perception, f, indent=2)
log("DONE ->", f"{OUT}/perception_eval.json")